# Phone Scrolling Detector

This version replaces phone detection with a completely separate phone-only model trained **from scratch**. The project now uses two detectors:

- a general detector for `face`, `person`, and `hand`
- a dedicated `mobile_phone` detector trained from scratch with phone-focused augmentation

That split is useful when dark phones or back-of-phone views are getting lost inside a broader multi-class model.


## Datasets

- **WIDER FACE** for `face`
- **Open Images V7** for `Person` and `Mobile phone`
- **EgoHands** for `hand`

The phone model uses only `Mobile phone` labels and is trained from scratch. To help with dark phones, the export step also creates darker and lower-contrast copies of phone-positive images.


## References

- WIDER FACE: https://mmlab.ie.cuhk.edu.hk/projects/WIDERFace/index.html
- Torchvision WIDERFace loader: https://docs.pytorch.org/vision/master/_modules/torchvision/datasets/widerface.html
- Open Images download page: https://storage.googleapis.com/openimages/web/download.html
- FiftyOne Open Images V7 docs: https://docs.voxel51.com/dataset_zoo/datasets/open_images_v7.html
- EgoHands official dataset page: https://vision.soic.indiana.edu/projects/egohands/


In [ ]:
%pip install -q ultralytics fiftyone gdown torch torchvision pyyaml tqdm pillow scipy numpy


In [ ]:
from __future__ import annotations

import random
import shutil
import zipfile
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import yaml
from PIL import Image, ImageDraw, ImageEnhance
from scipy.io import loadmat
from tqdm.auto import tqdm

import torch
from torchvision.datasets import WIDERFace

import fiftyone as fo
import fiftyone.zoo as foz

from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

@dataclass
class Config:
    project_root: Path = Path.cwd()
    data_root: Path = Path.cwd() / "data"
    raw_root: Path = Path.cwd() / "data" / "raw"
    general_root: Path = Path.cwd() / "data" / "scrolling_general_yolo"
    phone_root: Path = Path.cwd() / "data" / "scrolling_phone_yolo"
    runs_root: Path = Path.cwd() / "runs"
    open_images_train_samples: int = 1800
    open_images_val_samples: int = 300
    wider_train_limit: int = 9000
    wider_val_limit: int = 1800
    egohands_frame_stride: int = 3
    hand_val_fraction: float = 0.1
    general_epochs: int = 18
    phone_epochs: int = 30
    imgsz: int = 512
    batch: int = 24 if torch.cuda.is_available() else 8
    general_model_name: str = "yolo11n.pt"
    phone_model_yaml: str = "yolo11n.yaml"
    phone_dark_aug_copies: int = 1
    device: str = "0" if torch.cuda.is_available() else "cpu"

cfg = Config()

for path in [cfg.data_root, cfg.raw_root, cfg.general_root, cfg.phone_root, cfg.runs_root]:
    path.mkdir(parents=True, exist_ok=True)

GENERAL_CLASS_TO_ID = {"face": 0, "person": 1, "hand": 2}
PHONE_CLASS_TO_ID = {"mobile_phone": 0}

cfg


## Step 1: Download datasets

Download the official EgoHands labeled archive and place the zip or extracted folder under `data/raw/egohands` before running the conversion step. This notebook now fails fast if EgoHands is missing, so the `hand` class cannot be silently skipped.


In [ ]:
wider_train = WIDERFace(root=cfg.raw_root, split="train", download=True)
wider_val = WIDERFace(root=cfg.raw_root, split="val", download=True)

oi_train = foz.load_zoo_dataset(
    "open-images-v7",
    split="train",
    label_types=["detections"],
    classes=["Person", "Mobile phone"],
    max_samples=cfg.open_images_train_samples,
    shuffle=True,
    seed=SEED,
    dataset_name="open-images-v7-scroll-train",
)

oi_val = foz.load_zoo_dataset(
    "open-images-v7",
    split="validation",
    label_types=["detections"],
    classes=["Person", "Mobile phone"],
    max_samples=cfg.open_images_val_samples,
    shuffle=True,
    seed=SEED,
    dataset_name="open-images-v7-scroll-val",
)


## Step 2: Export two YOLO datasets

- `scrolling_general_yolo`: `face`, `person`, `hand`
- `scrolling_phone_yolo`: `mobile_phone` only

The phone export duplicates phone-positive samples and writes extra darkened versions to help with black or low-contrast phones.


In [ ]:
def reset_split_dirs(root: Path, split: str) -> tuple[Path, Path]:
    images_dir = root / "images" / split
    labels_dir = root / "labels" / split
    if images_dir.exists():
        shutil.rmtree(images_dir)
    if labels_dir.exists():
        shutil.rmtree(labels_dir)
    images_dir.mkdir(parents=True, exist_ok=True)
    labels_dir.mkdir(parents=True, exist_ok=True)
    return images_dir, labels_dir

def save_yolo_label(label_path: Path, rows: list[str]) -> None:
    label_path.write_text("\n".join(rows) + ("\n" if rows else ""), encoding="utf-8")

def xywh_to_yolo(x: float, y: float, w: float, h: float, img_w: int, img_h: int):
    if w <= 1 or h <= 1:
        return None
    x = max(0.0, min(x, img_w - 1))
    y = max(0.0, min(y, img_h - 1))
    w = max(1.0, min(w, img_w - x))
    h = max(1.0, min(h, img_h - y))
    return ((x + w / 2.0) / img_w, (y + h / 2.0) / img_h, w / img_w, h / img_h)

def dark_phone_variants(image: Image.Image) -> list[Image.Image]:
    variants = []
    settings = [
        (0.55, 0.85),
        (0.40, 0.75),
    ]
    for brightness_factor, contrast_factor in settings[: cfg.phone_dark_aug_copies]:
        dark = ImageEnhance.Brightness(image).enhance(brightness_factor)
        dark = ImageEnhance.Contrast(dark).enhance(contrast_factor)
        variants.append(dark)
    return variants

def find_fiftyone_detection_field(dataset: fo.Dataset) -> str:
    for field_name, field in dataset.get_field_schema().items():
        document_type = getattr(field, "document_type", None)
        if document_type is not None and document_type.__name__ == "Detections":
            return field_name
    raise RuntimeError("No FiftyOne detections field found")

def widerface_to_general(dataset: WIDERFace, split: str, images_dir: Path, labels_dir: Path, limit: int | None = None) -> int:
    written = 0
    img_info = dataset.img_info if limit is None else dataset.img_info[:limit]
    for idx, info in enumerate(tqdm(img_info, desc=f"WIDERFace -> {split}")):
        image_path = Path(info["img_path"])
        with Image.open(image_path) as im:
            img_w, img_h = im.size
        rows = []
        for x, y, w, h in info["annotations"]["bbox"].tolist():
            converted = xywh_to_yolo(float(x), float(y), float(w), float(h), img_w, img_h)
            if converted is None:
                continue
            xc, yc, wn, hn = converted
            rows.append(f"{GENERAL_CLASS_TO_ID['face']} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
        if not rows:
            continue
        target_name = f"wider_{split}_{idx:06d}{image_path.suffix.lower()}"
        shutil.copy2(image_path, images_dir / target_name)
        save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)
        written += 1
    return written

def discover_egohands_root(raw_root: Path) -> Path | None:
    candidates = [
        raw_root / "egohands",
        raw_root / "egohands" / "_LABELLED_SAMPLES",
        raw_root / "EgoHands",
        raw_root / "EgoHands" / "_LABELLED_SAMPLES",
    ]
    for path in candidates:
        if path.exists():
            return path
    zip_candidates = list(raw_root.rglob("*egohands*.zip")) + list(raw_root.rglob("*EgoHands*.zip"))
    if zip_candidates:
        extract_dir = raw_root / "egohands_extracted"
        if not extract_dir.exists():
            extract_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_candidates[0]) as zf:
                zf.extractall(extract_dir)
        for path in [extract_dir, extract_dir / "_LABELLED_SAMPLES"]:
            if path.exists():
                return path
        nested = list(extract_dir.rglob("_LABELLED_SAMPLES"))
        if nested:
            return nested[0]
    return None

def frame_number(path: Path) -> int:
    digits = "".join(ch for ch in path.stem if ch.isdigit())
    return int(digits) if digits else 0

def iter_polygons(node):
    if isinstance(node, np.ndarray):
        if node.dtype == object:
            for item in node.flat:
                yield from iter_polygons(item)
        else:
            arr = np.asarray(node)
            if arr.ndim == 2 and 2 in arr.shape and np.isfinite(arr).all():
                yield arr.astype(float) if arr.shape[1] == 2 else arr.T.astype(float)
    elif isinstance(node, (list, tuple)):
        for item in node:
            yield from iter_polygons(item)

def polygon_to_bbox(poly: np.ndarray, img_w: int, img_h: int):
    if poly.size == 0:
        return None
    xs = np.clip(poly[:, 0], 0, img_w - 1)
    ys = np.clip(poly[:, 1], 0, img_h - 1)
    x1, x2 = float(xs.min()), float(xs.max())
    y1, y2 = float(ys.min()), float(ys.max())
    if x2 - x1 < 2 or y2 - y1 < 2:
        return None
    return x1, y1, x2 - x1, y2 - y1

def egohands_to_general(root: Path | None, train_images: Path, train_labels: Path, val_images: Path, val_labels: Path) -> dict:
    if root is None:
        raise FileNotFoundError("EgoHands was not found under data/raw. Put the official labeled archive under data/raw/egohands before training.")
    if (root / "_LABELLED_SAMPLES").exists():
        root = root / "_LABELLED_SAMPLES"
    locations = sorted([p for p in root.iterdir() if p.is_dir()])
    train_written = 0
    val_written = 0
    modulo = max(1, round(1 / cfg.hand_val_fraction))
    for location_idx, location_dir in enumerate(tqdm(locations, desc="EgoHands")):
        polygons_path = location_dir / "polygons.mat"
        frame_paths = sorted(location_dir.glob("*.jpg"), key=frame_number)
        if not polygons_path.exists() or not frame_paths:
            continue
        polygons = loadmat(polygons_path).get("polygons")
        if polygons is None:
            continue
        use_val = (location_idx % modulo) == 0
        images_dir = val_images if use_val else train_images
        labels_dir = val_labels if use_val else train_labels
        for frame_idx in range(0, min(len(frame_paths), len(polygons)), cfg.egohands_frame_stride):
            image_path = frame_paths[frame_idx]
            with Image.open(image_path) as im:
                img_w, img_h = im.size
            rows = []
            for poly in iter_polygons(polygons[frame_idx]):
                bbox = polygon_to_bbox(poly, img_w, img_h)
                if bbox is None:
                    continue
                converted = xywh_to_yolo(*bbox, img_w, img_h)
                if converted is None:
                    continue
                xc, yc, wn, hn = converted
                rows.append(f"{GENERAL_CLASS_TO_ID['hand']} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
            if not rows:
                continue
            target_name = f"egohands_{'val' if use_val else 'train'}_{location_idx:03d}_{frame_idx:04d}.jpg"
            shutil.copy2(image_path, images_dir / target_name)
            save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)
            if use_val:
                val_written += 1
            else:
                train_written += 1
    return {"train": train_written, "val": val_written}

def openimages_to_general(dataset: fo.Dataset, split: str, images_dir: Path, labels_dir: Path) -> int:
    det_field = find_fiftyone_detection_field(dataset)
    class_map = {"Person": GENERAL_CLASS_TO_ID['person']}
    written = 0
    for idx, sample in enumerate(tqdm(dataset.iter_samples(progress=True, autosave=False), desc=f"OpenImages general -> {split}")):
        detections = getattr(sample, det_field, None)
        if detections is None or not detections.detections:
            continue
        rows = []
        for det in detections.detections:
            if det.label not in class_map:
                continue
            x, y, w, h = det.bounding_box
            rows.append(f"{class_map[det.label]} {x + w/2:.6f} {y + h/2:.6f} {w:.6f} {h:.6f}")
        if not rows:
            continue
        image_path = Path(sample.filepath)
        target_name = f"oi_general_{split}_{idx:06d}{image_path.suffix.lower()}"
        shutil.copy2(image_path, images_dir / target_name)
        save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", rows)
        written += 1
    return written

def openimages_to_phone(dataset: fo.Dataset, split: str, images_dir: Path, labels_dir: Path) -> int:
    det_field = find_fiftyone_detection_field(dataset)
    written = 0
    for idx, sample in enumerate(tqdm(dataset.iter_samples(progress=True, autosave=False), desc=f"OpenImages phone -> {split}")):
        detections = getattr(sample, det_field, None)
        if detections is None or not detections.detections:
            continue
        phone_rows = []
        for det in detections.detections:
            if det.label != "Mobile phone":
                continue
            x, y, w, h = det.bounding_box
            phone_rows.append(f"{PHONE_CLASS_TO_ID['mobile_phone']} {x + w/2:.6f} {y + h/2:.6f} {w:.6f} {h:.6f}")
        if not phone_rows:
            continue
        image_path = Path(sample.filepath)
        with Image.open(image_path).convert("RGB") as im:
            target_name = f"oi_phone_{split}_{idx:06d}.jpg"
            im.save(images_dir / target_name, quality=95)
            save_yolo_label(labels_dir / f"{Path(target_name).stem}.txt", phone_rows)
            written += 1
            if split == "train":
                for aug_idx, aug_im in enumerate(dark_phone_variants(im)):
                    aug_name = f"oi_phone_{split}_{idx:06d}_dark{aug_idx}.jpg"
                    aug_im.save(images_dir / aug_name, quality=95)
                    save_yolo_label(labels_dir / f"{Path(aug_name).stem}.txt", phone_rows)
                    written += 1
    return written

general_train_images, general_train_labels = reset_split_dirs(cfg.general_root, "train")
general_val_images, general_val_labels = reset_split_dirs(cfg.general_root, "val")
phone_train_images, phone_train_labels = reset_split_dirs(cfg.phone_root, "train")
phone_val_images, phone_val_labels = reset_split_dirs(cfg.phone_root, "val")

egohands_root = discover_egohands_root(cfg.raw_root)
wider_train_written = widerface_to_general(wider_train, "train", general_train_images, general_train_labels, limit=cfg.wider_train_limit)
wider_val_written = widerface_to_general(wider_val, "val", general_val_images, general_val_labels, limit=cfg.wider_val_limit)
oi_general_train_written = openimages_to_general(oi_train, "train", general_train_images, general_train_labels)
oi_general_val_written = openimages_to_general(oi_val, "val", general_val_images, general_val_labels)
oi_phone_train_written = openimages_to_phone(oi_train, "train", phone_train_images, phone_train_labels)
oi_phone_val_written = openimages_to_phone(oi_val, "val", phone_val_images, phone_val_labels)
egohands_written = egohands_to_general(egohands_root, general_train_images, general_train_labels, general_val_images, general_val_labels)
if egohands_written['train'] == 0:
    raise RuntimeError("EgoHands export completed but produced 0 training samples, so the hand class still is not being trained.")

general_yaml_path = cfg.general_root / "general.yaml"
general_yaml_path.write_text(yaml.safe_dump({
    "path": str(cfg.general_root.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": ["face", "person", "hand"],
}, sort_keys=False), encoding="utf-8")

phone_yaml_path = cfg.phone_root / "phone.yaml"
phone_yaml_path.write_text(yaml.safe_dump({
    "path": str(cfg.phone_root.resolve()),
    "train": "images/train",
    "val": "images/val",
    "names": ["mobile_phone"],
}, sort_keys=False), encoding="utf-8")

print({
    "wider_train": wider_train_written,
    "wider_val": wider_val_written,
    "oi_general_train": oi_general_train_written,
    "oi_general_val": oi_general_val_written,
    "oi_phone_train": oi_phone_train_written,
    "oi_phone_val": oi_phone_val_written,
    "egohands_train": egohands_written['train'],
    "egohands_val": egohands_written['val'],
})


## Step 3: Train the two detectors

The general detector can still use transfer learning. The phone detector is intentionally trained **from scratch** by starting from a model YAML instead of a `.pt` checkpoint and setting `pretrained=False`.


In [ ]:
general_detector = YOLO(cfg.general_model_name)
general_detector.train(
    data=str(general_yaml_path),
    epochs=cfg.general_epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    device=cfg.device,
    project=str(cfg.runs_root),
    name="scrolling_general_detector_fast",
    exist_ok=True,
    pretrained=True,
    workers=2,
    patience=6,
    cache=False,
)
general_best_weights = Path(general_detector.trainer.best)

phone_detector = YOLO(cfg.phone_model_yaml)
phone_detector.train(
    data=str(phone_yaml_path),
    epochs=cfg.phone_epochs,
    imgsz=cfg.imgsz,
    batch=cfg.batch,
    device=cfg.device,
    project=str(cfg.runs_root),
    name="scrolling_phone_detector_scratch_fast",
    exist_ok=True,
    pretrained=False,
    workers=2,
    patience=8,
    cache=False,
    hsv_h=0.02,
    hsv_s=0.70,
    hsv_v=0.65,
    degrees=18.0,
    translate=0.10,
    scale=0.60,
    shear=3.0,
    perspective=0.0010,
    fliplr=0.5,
    mosaic=0.7,
    mixup=0.10,
    copy_paste=0.15,
    erasing=0.20,
    close_mosaic=6,
)
phone_best_weights = Path(phone_detector.trainer.best)

general_best_weights, phone_best_weights


## Step 4: Merge the detectors for scrolling inference

During inference we run both models, then combine the detections. The scrolling score prefers phone boxes that line up with hands and a plausible face-to-phone position.


In [ ]:
general_model = YOLO(str(general_best_weights))
phone_model = YOLO(str(phone_best_weights))

def xyxy_center(box):
    x1, y1, x2, y2 = box
    return ((x1 + x2) / 2.0, (y1 + y2) / 2.0)

def box_size(box):
    x1, y1, x2, y2 = box
    return (x2 - x1, y2 - y1)

def contains_point(box, point):
    x1, y1, x2, y2 = box
    px, py = point
    return x1 <= px <= x2 and y1 <= py <= y2

def expand_box(box, frac):
    x1, y1, x2, y2 = box
    w, h = x2 - x1, y2 - y1
    return [x1 - w * frac, y1 - h * frac, x2 + w * frac, y2 + h * frac]

def iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    if inter <= 0:
        return 0.0
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def result_to_named(result, label_map, conf):
    output = {label: [] for label in label_map.values()}
    for cls_id, score, xyxy in zip(result.boxes.cls.tolist(), result.boxes.conf.tolist(), result.boxes.xyxy.tolist()):
        if score < conf:
            continue
        label = label_map[int(cls_id)]
        output[label].append({"box": xyxy, "score": float(score)})
    return output

def predict_scrolling(image_path, general_conf=0.20, phone_conf=0.15, scroll_threshold=0.55):
    general_result = general_model.predict(source=str(image_path), conf=general_conf, verbose=False)[0]
    phone_result = phone_model.predict(source=str(image_path), conf=phone_conf, verbose=False)[0]
    general_named = result_to_named(general_result, {0: 'face', 1: 'person', 2: 'hand'}, general_conf)
    phone_named = result_to_named(phone_result, {0: 'mobile_phone'}, phone_conf)
    named = {
        'face': general_named['face'],
        'person': general_named['person'],
        'hand': general_named['hand'],
        'mobile_phone': phone_named['mobile_phone'],
    }

    faces = named['face']
    phones = named['mobile_phone']
    persons = named['person']
    hands = named['hand']
    if not faces or not phones:
        return {'image_path': str(image_path), 'is_scrolling': False, 'scroll_score': 0.0, 'detections': named, 'meta': {'reason': 'missing_face_or_phone'}}

    best_score = 0.0
    best_meta = {'reason': 'no_match'}
    for face in faces:
        face_box = face['box']
        face_cx, face_cy = xyxy_center(face_box)
        face_w, face_h = box_size(face_box)
        candidate_people = [p for p in persons if contains_point(p['box'], (face_cx, face_cy))] or persons or [{'box': [face_box[0] - 2 * face_w, face_box[1], face_box[2] + 2 * face_w, face_box[3] + 6 * face_h]}]
        for person in candidate_people:
            person_box = person['box']
            person_w, person_h = box_size(person_box)
            if person_w <= 1 or person_h <= 1:
                continue
            person_hands = [h for h in hands if contains_point(person_box, xyxy_center(h['box']))]
            for phone in phones:
                phone_box = phone['box']
                phone_cx, phone_cy = xyxy_center(phone_box)
                phone_w, phone_h = box_size(phone_box)
                if not contains_point(person_box, (phone_cx, phone_cy)):
                    continue
                support_hands = [h for h in person_hands if iou(expand_box(h['box'], 0.35), expand_box(phone_box, 0.20)) > 0.0]
                horizontal_distance = abs(phone_cx - face_cx) / max(person_w, 1.0)
                vertical_offset = (phone_cy - face_cy) / max(person_h, 1.0)
                phone_scale = max(phone_w, phone_h) / max(person_w, 1.0)
                score = 0.0
                score += 0.22 if horizontal_distance <= 0.38 else max(0.0, 0.22 - horizontal_distance)
                score += 0.22 if 0.02 <= vertical_offset <= 0.48 else max(0.0, 0.22 - abs(vertical_offset - 0.18))
                score += 0.12 if 0.02 <= phone_scale <= 0.32 else max(0.0, 0.12 - abs(phone_scale - 0.10))
                score += 0.24 if len(support_hands) >= 1 else 0.0
                score += 0.12 if len(support_hands) >= 2 else 0.0
                score += 0.08 * min(face['score'], phone['score'])
                if score > best_score:
                    best_score = min(float(score), 1.0)
                    best_meta = {
                        'face_box': face_box,
                        'person_box': person_box,
                        'phone_box': phone_box,
                        'supporting_hands': [h['box'] for h in support_hands],
                    }
    return {'image_path': str(image_path), 'is_scrolling': best_score >= scroll_threshold, 'scroll_score': round(best_score, 4), 'detections': named, 'meta': best_meta}


In [ ]:
TEST_IMAGE = None

if TEST_IMAGE is not None:
    prediction = predict_scrolling(TEST_IMAGE)
    print(prediction['is_scrolling'], prediction['scroll_score'])
    print(prediction['meta'])
else:
    print('Set TEST_IMAGE to a local path before running this cell.')


In [ ]:
def draw_prediction(image_path, prediction):
    im = Image.open(image_path).convert('RGB')
    draw = ImageDraw.Draw(im)
    color_map = {'face': 'lime', 'person': 'cyan', 'hand': 'magenta', 'mobile_phone': 'yellow'}
    for label, detections in prediction['detections'].items():
        for det in detections:
            x1, y1, x2, y2 = det['box']
            draw.rectangle([x1, y1, x2, y2], outline=color_map[label], width=3)
            draw.text((x1 + 4, max(0, y1 - 14)), f"{label}:{det['score']:.2f}", fill=color_map[label])
    draw.text((12, 12), f"scrolling={prediction['is_scrolling']} score={prediction['scroll_score']:.2f}", fill='white')
    return im

if TEST_IMAGE is not None:
    draw_prediction(TEST_IMAGE, prediction)


## Notes

The phone detector is now the piece that should improve most for dark phones, because:

- it is trained only on phones
- it starts from scratch instead of inheriting generic object biases
- it sees synthetic darker phone examples during export
- it uses stronger brightness and saturation augmentation during training

If dark phones are still weak after retraining, the next best improvement is to add your own hard examples from the same camera angle and lighting conditions.
